In [1]:
from nero_interface_server import NeroDualArmServer

In [2]:
import logging
log = logging.getLogger(__name__)

In [3]:
import numpy as np
import time

In [4]:
server = NeroDualArmServer(gripper_enabled=True)

[SERVER] Failed to connect to right arm: CAN socket can_right does not exist.


[Left Arm] 正在获取当前关节角作为 IK 初始基准...
[Left Arm] IK Solver 初始化完成！初始关节角: [-0.299  1.085  0.206  2.192 -0.181 -0.073 -1.697]


In [ ]:
fp = server.left_robot.get_flange_pose()
robot_pose = np.array(fp.msg)

print("robot z:", robot_pose[2])

In [ ]:
assert server.left_robot is not None, "Left arm failed"
assert server.right_robot is not None, "Right arm failed"
print("Left robot:", server.left_robot)
print("Right robot:", server.right_robot)

In [ ]:
left_joints = server.left_robot_get_joint_positions()
right_joints = server.right_robot_get_joint_positions()

print("Left joints:", left_joints)
print("Right joints:", right_joints)

In [ ]:
left_pose = server.left_robot_get_ee_pose()
right_pose = server.right_robot_get_ee_pose()
print("Left pose:", left_pose)
print("Right pose:", right_pose)

In [ ]:
left_arm_status = server.left_robot_get_arm_status()
right_arm_status = server.right_robot_get_arm_status()

print("Left arm status:", left_arm_status)
print("Right arm status:", right_arm_status)

In [25]:
server.left_robot_go_home()

left_joints = server.left_robot_get_joint_positions()
print("Left joints:", left_joints)

Left joints: [0.0, -0.12999212268853766, 1.7453292519943296e-05, 1.8699632138792448, 0.0, -0.00013962634015954637, -0.1699950691442477]


In [ ]:
tcp = server.left_robot.get_tcp_pose()
print(tcp.msg)

In [ ]:
# left_joints = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints)

# left_joints_target = np.array(left_joints).copy()
# left_joints_target[0] -= np.deg2rad(0)
# left_joints_target[1] += np.deg2rad(0)
# left_joints_target[2] += np.deg2rad(0)
# left_joints_target[3] += np.deg2rad(0)
# left_joints_target[4] += np.deg2rad(0)
# left_joints_target[5] += np.deg2rad(0)
# left_joints_target[6] += np.deg2rad(0)

# print("Left joints target:", left_joints_target)

# server.left_robot_move_to_joint_positions(left_joints_target, delta=False)

# left_joints_real = server.left_robot_get_joint_positions()
# print("Left joints:", left_joints_real)

In [ ]:
# left_pose = server.left_robot_get_ee_pose()
# print("Left pose:", left_pose)

# pose_target = np.array(left_pose).copy()
# # end-effector pose [x, y, z, roll, pitch, yaw] (m, rad)
# # pose_target[0] += 0.01   # x方向移动1cm
# # pose_target[1] += 0.01
# # pose_target[2] += 0.01
# pose_target = [-0.4, 0.0, 0.4, 1.5708, 0.0, 0.0]

# server.left_robot_move_to_ee_pose(pose_target)
# time.sleep(3.0)
# left_pose = server.left_robot_get_ee_pose()
# print("Left pose:", left_pose)

In [20]:
# 绝对控制
target = [0, -70, 0, 100, 0, 0, 30]  # 7维绝对关节角度（度）

ok = server.servo_j("left_robot", target, delta=False)
print("servo_j ok:", ok)

servo_j ok: True


In [ ]:
# TODO: bug
# # 增量控制
# target_delta = np.zeros(7)
# target_delta[0] = 10

# ok = server.servo_j("left_robot", target_delta.tolist(), delta=True)
# print("servo_j_delta ok:", ok)

servo_j_delta ok: True


In [ ]:
# 当前位姿
fp = server.left_robot.get_flange_pose()
current_pose = np.array(fp.msg, dtype=float)
print("ipynb当前位姿:", current_pose)

# 构造一个小的 delta
delta_pose = np.array([0.05, 0.0, 0.0, 0.0, 0.0, 0.0])
# target_pose = np.array([-0.5, 0.05, 0.2, 1.46, -0.7, -0.04])

# 调用 servo_p（delta 模式）
ret = server.servo_p(
    robot_arm="left_robot",
    pose=delta_pose.tolist(),
    delta=True
    # pose=target_pose.tolist(),
    # delta=False
)

time.sleep(1.5)

fp = server.left_robot.get_flange_pose()
new_pose = np.array(fp.msg, dtype=float)
print("新位姿:", new_pose)

In [ ]:
# 当前位姿
fp = server.left_robot.get_flange_pose()
current_pose = np.array(fp.msg, dtype=float)
# print("当前位姿:", current_pose)

target_pose = current_pose.copy()
# target_pose[0] += 0.01
# print("目标位姿:", target_pose)

steps = 10
dt = 0.02  # 20Hz

# for i in range(steps):
#     alpha = (i + 1) / steps
#     pose_i = current_pose * (1 - alpha) + target_pose * alpha

#     server.servo_p("left_robot", pose_i.tolist(), delta=False)
#     time.sleep(dt)

for i in range(steps):
    # target_pose[2] += 0.02 / steps   # 每次走一点点
    delta_pose = np.array([-0.01, 0.0, 0.0, 0.0, 0.0, 0.0])
    time1 = time.time()
    server.servo_p("left_robot", delta_pose.tolist(), delta=True)
    time2 = time.time()
    print(f"Step {i+1}/{steps}, servo_p time: {(time2 - time1) * 1000:.2f} ms")
    time.sleep(0.005)

In [ ]:
# server.left_gripper_goto(0.05)
# # server.left_gripper_grasp()
# print(server.left_gripper_get_state())

In [ ]:
# server.stop("left_robot")